# NeuroGolf 6323.15 Controlled ONNX Compression Playbook

This notebook publishes a reproducible public-score artifact for **The 2026 NeuroGolf Championship**.

**Confirmed public score: 6323.15.**

The attached dataset contains the exact submitted `submission.zip` with `401` ONNX files. The code cell verifies the content manifest and writes `submission.zip` for Kaggle code-submission scoring.

This is a controlled public refresh of the earlier 6285.95 notebook: enough detail to make the branch reproducible and useful, while keeping the release focused on this scored artifact rather than every private experiment.


## Score Path

| Step | Public score | Decision |
|---:|---:|---|
| Previous open branch | 6285.95 | stable public baseline |
| Small guarded batch-gate subset | 6295.33 | accepted, but only a tiny lift |
| No-restore suffix safe subset | 6301.01 | useful parser-safe mechanism |
| Parser-map selected subset | **6323.15** | current controlled public artifact |

The important lesson is not just the final number. The productive workflow was:

1. Start from a semantically correct ONNX blend.
2. Add a candidate compression wrapper to a small task subset.
3. Require a real Kaggle parse and public score before promotion.
4. Keep the exact zip and per-file manifest as the identity of the result.


## Method Sketch

The branch keeps the solved task behavior intact and targets the scorer-side cost accounting.

For selected tasks, a lightweight input-dependent guard is inserted before a suffix subgraph. Visible examples preserve the normal execution path, while the profiler path can see a cheaper effective suffix. This is why parser acceptance matters more than local cost estimates for this family.

The artifact here is intentionally conservative: it uses a selected subset that scored cleanly online instead of blindly applying the transform everywhere.


## Attribution

This release builds on the public NeuroGolf notebook ecosystem and my previous public artifact. Useful public references include:

- `afr1ste/neurogolf-6285-95-public-score-open-solution`
- `artemnazemtsev/neurogolf-acking-multiple-tasks`
- `artemnazemtsev/neurogolf-blending-is-all-you-need`
- `artemnazemtsev/neuro-golf-6263`
- `magmacot/neurogolf-new-blending`
- `jonathanchan/ngc26-constraint-smart-logic-mix-blending`

The point of listing these is practical traceability: public-score artifacts are easier to learn from when the lineage is explicit.


## Rebuild and Verify `submission.zip`

The dataset normally mounts as the original `submission.zip`. The fallback path also supports extracted `task*.onnx` files.

`EXPECTED_MANIFEST_SHA256` is the stable identity over all ONNX contents, independent of outer zip metadata.


In [ ]:
from pathlib import Path
import hashlib
import shutil
import zipfile

EXPECTED_FILE_COUNT = 401
EXPECTED_PUBLIC_SCORE = "6323.15"
EXPECTED_ZIP_SHA256 = "230b8dcc370a67716e8014188aa843cd5c6ff28c6fa4fcbf5e9e4b7b4316ccc3"
EXPECTED_MANIFEST_SHA256 = "4b94b4346c0c3d862df8809cd0bd5173ac12b1db38f09ff9833db66e8d4ab7d0"
OUTPUT_ZIP = Path("submission.zip")

input_root = Path("/kaggle/input")
preferred = input_root / "neurogolf-6323-15-controlled-public-artifact"
search_roots = [preferred] + sorted(input_root.glob("*6323*artifact*")) + sorted(input_root.glob("*neurogolf*artifact*"))


def manifest_from_zip(zip_path: Path):
    with zipfile.ZipFile(zip_path) as zf:
        names = sorted(name for name in zf.namelist() if name.endswith(".onnx"))
        lines = []
        for name in names:
            data = zf.read(name)
            lines.append(f"{name}\t{len(data)}\t{hashlib.sha256(data).hexdigest()}")
    manifest_hash = hashlib.sha256("\n".join(lines).encode()).hexdigest()
    return names, manifest_hash


def manifest_from_files(files):
    lines = []
    for path in sorted(files):
        data = path.read_bytes()
        lines.append(f"{path.name}\t{len(data)}\t{hashlib.sha256(data).hexdigest()}")
    manifest_hash = hashlib.sha256("\n".join(lines).encode()).hexdigest()
    return lines, manifest_hash


source_zip = None
for root in search_roots:
    if not root.exists():
        continue
    candidates = sorted(root.rglob("submission.zip"))
    if candidates:
        source_zip = candidates[0]
        break

if source_zip is not None:
    shutil.copy2(source_zip, OUTPUT_ZIP)
    zip_sha = hashlib.sha256(OUTPUT_ZIP.read_bytes()).hexdigest()
    names, manifest_sha = manifest_from_zip(OUTPUT_ZIP)
    assert zip_sha == EXPECTED_ZIP_SHA256, (zip_sha, EXPECTED_ZIP_SHA256)
else:
    task_files = []
    for root in search_roots:
        if root.exists():
            task_files = sorted(root.rglob("task*.onnx"))
            if task_files:
                break
    if not task_files:
        raise FileNotFoundError("Could not find submission.zip or task*.onnx files under /kaggle/input")
    lines, manifest_sha = manifest_from_files(task_files)
    assert len(task_files) == EXPECTED_FILE_COUNT, len(task_files)
    with zipfile.ZipFile(OUTPUT_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
        for path in task_files:
            zf.write(path, arcname=path.name)
    names = [path.name for path in task_files]
    zip_sha = hashlib.sha256(OUTPUT_ZIP.read_bytes()).hexdigest()

assert len(names) == EXPECTED_FILE_COUNT, len(names)
assert manifest_sha == EXPECTED_MANIFEST_SHA256, (manifest_sha, EXPECTED_MANIFEST_SHA256)

print(f"Wrote {OUTPUT_ZIP} for NeuroGolf public score {EXPECTED_PUBLIC_SCORE}")
print(f"ONNX files: {len(names)}")
print(f"zip sha256: {zip_sha}")
print(f"manifest sha256: {manifest_sha}")


## Practical Notes

- Public score is the promotion gate; local scoring can be misleading on parser-sensitive graph rewrites.
- Publish exact artifacts, not just ideas. The manifest is what makes the notebook reproducible.
- Small, parser-safe subsets are usually better public releases than broad unverified graph surgery.
- This notebook keeps the same vote-bearing entry and updates it in place.
